In [15]:
import os
import uuid
from langchain_core.tools import tool
from langchain_openai import ChatOpenAI
from langchain.agents import create_agent
from langgraph.store.memory import InMemoryStore
from dotenv import load_dotenv

load_dotenv()

OPENAI_API_KEY = os.getenv("OPENROUTER_API_KEY")

model = ChatOpenAI(
    api_key=OPENAI_API_KEY,
    base_url="https://openrouter.ai/api/v1",
    model="openai/gpt-4o-mini"
)

store = InMemoryStore()

@tool
def add_task(task: str) -> str:
    """Додає нове завдання до списку."""
    task_id = str(uuid.uuid4())
    store.put(("tasks",), task_id, {"id": task_id, "text": task})
    return f"Завдання '{task}' додано!"


@tool
def list_tasks() -> str:
    """Показує всі завдання зі списку."""
    items = store.search(('tasks',))
    if not items:
        return "Список завдань порожній."
    return "\n".join([f"{i}. {item.value['text']} [{item.value['id'][:8]}]"
                  for i, item in enumerate(items, 1)])


# TODO: Створіть інструмент для видалення завдання
@tool
def delete_task(task_id: str) -> str:
    """Видаляє завдання за id."""
    item = store.get(("tasks",), task_id)
    if not item:
        return f"Завдання з id {task_id} не знайдено"
    store.delete(("tasks",), task_id)
    return f"Завдання '{item.value['text']}' видалено"


# TODO: Створіть агента з інструментами та store
agent = create_agent(
    model,
    tools=[add_task, list_tasks, delete_task],
    store=store,
    system_prompt="Ти помічник для управління завданнями. Відповідай українською мовою."
)


# TODO: Протестуйте агента
tests = [
    "Додай: купити хліб",
    "Додай: подзвонити лікарю",
    "Покажи всі завдання",
    "Видали завдання про хліб",
    "Що залишилось?"
]


for i, test in enumerate(tests, 1):
    print(f"\n{'='*50}")
    print(f"TEST {i}: {test}")
    print('='*50)
    res = agent.invoke({"messages": [{"role": "user", "content": test}]})
    print(res["messages"][-1].content)




TEST 1: Додай: купити хліб
Завдання "купити хліб" успішно додано до списку!

TEST 2: Додай: подзвонити лікарю
Завдання "подзвонити лікарю" додано! Чи потрібно ще щось зробити?

TEST 3: Покажи всі завдання
Ось всі твої завдання:

1. купити хліб [9e641e5c]
2. подзвонити лікарю [b4fbffa4]

TEST 4: Видали завдання про хліб
Схоже, що завдання про хліб не вдалося видалити, оскільки воно не знайдено. Можливо, завдання було видалено раніше. Якщо потрібно, я можу допомогти з іншими завданнями.

TEST 5: Що залишилось?
У вас залишились такі завдання:
1. Купити хліб
2. Подзвонити лікарю
